In [102]:
import pandas as pd
import numpy as np
import networkx as nx
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.preprocessing import scale
from scipy.stats import zscore

In [88]:
# Step 1: Load the Data
data = pd.read_csv("GSE18123-GPL6244_series_matrix.txt", sep="\t", index_col=0)

In [89]:
print(data.head())

                                                diagnosis: ASPERGER'S DISORDER  \
!Sample_characteristics_ch1                                                      
!Sample_characteristics_ch1                                       gender: male   
!Sample_characteristics_ch1                 age at blood drawing (months): 156   
!Sample_characteristics_ch1                                    race: Caucasian   
!Sample_characteristics_ch1                            ethnicity: Non-Hispanic   
!Sample_characteristics_ch1  hours (minutes since last caloric intake): : 3:25   

                                                             diagnosis: AUTISM  \
!Sample_characteristics_ch1                                                      
!Sample_characteristics_ch1                                       gender: male   
!Sample_characteristics_ch1                 age at blood drawing (months): 168   
!Sample_characteristics_ch1                                    race: Caucasian   
!Sample_charact

In [90]:
print(data.columns)

Index(['diagnosis: ASPERGER'S DISORDER', 'diagnosis: AUTISM',
       'diagnosis: CONTROL', 'diagnosis: AUTISM.1', 'diagnosis: AUTISM.2',
       'diagnosis: PDD-NOS', 'diagnosis: PDD-NOS.1',
       'diagnosis: ASPERGER'S DISORDER.1', 'diagnosis: PDD-NOS.2',
       'diagnosis: PDD-NOS.3',
       ...
       'diagnosis: CONTROL.72', 'diagnosis: CONTROL.73',
       'diagnosis: CONTROL.74', 'diagnosis: CONTROL.75',
       'diagnosis: CONTROL.76', 'diagnosis: CONTROL.77',
       'diagnosis: CONTROL.78', 'diagnosis: CONTROL.79',
       'diagnosis: CONTROL.80', 'diagnosis: CONTROL.81'],
      dtype='object', length=186)


In [106]:
ESA = pd.read_csv("12920_2023_1439_MOESM1_ESM.csv")
ESC = pd.read_csv("12920_2023_1439_MOESM2_ESM.csv")
G_ppi = pd.read_csv("12920_2023_1439_MOESM3_ESM.csv")
G_miR = pd.read_csv("12920_2023_1439_MOESM4_ESM.csv")

In [107]:
ESA.head()

,Unnamed: 0,GSM650659.CEL.gz,GSM650676.CEL.gz,GSM650679.CEL.gz,GSM650686.CEL.gz,GSM650691.CEL.gz,GSM650692.CEL.gz,GSM650696.CEL.gz,GSM650709.CEL.gz,GSM650713.CEL.gz,...,GSM650725.CEL.gz,GSM650728.CEL.gz,GSM650729.CEL.gz,GSM650826.CEL.gz,GSM650828.CEL.gz,GSM650835.CEL.gz,GSM650850.CEL.gz,GSM650855.CEL.gz,GSM650856.CEL.gz,GSM650869.CEL.gz
0,LINC01128,6.906442,7.198315,7.186099,7.296602,6.808630,6.999142,6.981611,6.834657,7.249340,...,7.121044,7.504075,7.420045,6.931073,7.432651,7.400328,7.510892,7.411977,7.581309,7.732369
1,ISG15,7.818835,7.653704,8.251708,8.035551,7.931100,8.297684,8.299747,7.942194,7.843039,...,7.907336,7.914256,8.030161,7.292140,6.924464,7.551829,7.715360,6.954788,7.310003,8.238479
2,SLC25A33,8.695718,8.709581,8.708811,8.459994,8.650054,8.387098,8.567997,8.829396,8.561444,...,8.359148,8.644014,8.742646,8.522925,8.372858,8.122176,8.027500,8.031494,8.306064,8.310499
3,RBP7,6.481948,6.665605,7.065216,6.349913,7.499084,6.973699,7.058077,6.405605,6.912880,...,6.200864,6.810367,7.180123,6.910874,7.215533,6.611417,6.667153,7.458872,6.801707,6.626787
4,PGD,9.810405,10.507621,10.139465,10.319822,10.340524,10.436695,10.416218,10.087888,9.902462,...,9.397798,10.229169,9.551669,10.546762,10.244057,9.876214,10.633884,10.443538,9.829885,10.570067


In [108]:
ESC.head()

,Unnamed: 0,GSM650876.CEL.gz,GSM650877.CEL.gz,GSM650878.CEL.gz,GSM650879.CEL.gz,GSM650880.CEL.gz,GSM650883.CEL.gz,GSM650885.CEL.gz,GSM650923.CEL.gz,GSM650924.CEL.gz,...,GSM650944.CEL.gz,GSM650946.CEL.gz,GSM650950.CEL.gz,GSM650951.CEL.gz,GSM650954.CEL.gz,GSM650959.CEL.gz,GSM650968.CEL.gz,GSM650969.CEL.gz,GSM650970.CEL.gz,GSM650975.CEL.gz
0,LINC01128,7.575569,7.817170,7.695029,7.495486,7.462981,7.086119,7.187736,7.155224,7.489574,...,7.515678,7.907138,8.179843,7.881976,7.736908,7.800728,7.591349,7.927571,7.807398,8.057399
1,ISG15,7.624894,7.117802,7.520475,7.474655,7.412729,7.167821,7.685093,7.735854,7.190735,...,8.699819,7.373696,7.414944,7.302315,7.404726,7.685369,7.509189,7.263457,7.344701,7.536274
2,SLC25A33,8.693018,8.428586,8.340659,8.281868,8.269921,7.873420,8.462543,8.360530,7.973997,...,7.594758,7.780251,7.291135,7.469125,7.744339,7.779572,7.807796,7.575769,7.436038,7.669425
3,RBP7,6.947236,6.634827,5.756420,5.909541,6.397363,6.477703,6.159352,6.675923,6.397966,...,6.298439,6.546234,5.860817,6.261531,6.328678,5.939276,6.493185,6.298535,6.393383,6.241520
4,PGD,10.159275,10.304521,10.166557,9.074119,10.255038,10.166952,10.099866,10.138683,10.917744,...,10.934496,10.781862,10.078060,10.717894,10.852050,10.657963,10.796962,11.048911,10.859075,10.775936


In [109]:
G_ppi.head()

,Unnamed: 0,from,to,combined_score
0,1,9606.ENSP00000008938,9606.ENSP00000011653,208
1,2,9606.ENSP00000008938,9606.ENSP00000011653,208
2,3,9606.ENSP00000009530,9606.ENSP00000011653,408
3,4,9606.ENSP00000009530,9606.ENSP00000011653,408
4,5,9606.ENSP00000009530,9606.ENSP00000085219,524


In [110]:
G_miR.head()

,ID,Accession,Target,TargetID,Experiment,Literature,Tissue
0,hsa-mir-17-5p,MIMAT0000070,CDKN1A,1026,HITS-CLIP//Luciferase reporter assay//Northern...,18212054|18493594|20190813|18941111|20227518|2...,Peripheral blood
1,hsa-mir-17-5p,MIMAT0000070,CCL5,6352,HITS-CLIP,23313552,Peripheral blood
2,hsa-mir-17-5p,MIMAT0000070,TNF,7124,Luciferase reporter assay,26041742,Peripheral blood
3,hsa-mir-17-5p,MIMAT0000070,TP53,7157,qRT-PCR,24955218,Peripheral blood
4,hsa-mir-17-5p,MIMAT0000070,TLR7,51284,HITS-CLIP//Luciferase reporter assay,23313552|26041742,Peripheral blood


In [111]:
autism.shape

(4701, 22)

In [112]:
g = pd.read_excel("G.xlsx")

In [113]:
g.shape

(33319, 187)

In [114]:
def FA_gene(G, ESA, ESC):
    
    # Step 2: Reduce gene set G using variance function (Var(.))
    G_reduced = G[G.var(axis=0) >= 0.75]

    # Step 3: Generate gene expression matrices ESA_G and ESC_G based on reduced gene set G_reduced
    ESA_G = ESA[G_reduced.index]
    ESC_G = ESC[G_reduced.index]

    # Step 4: Construct co-expression network for control samples (ESC_G)
    # For simplicity, we will use networkx
    coexp_network = nx.Graph()
    # Calculate correlation matrix
    correlation_matrix = ESC_G.corr()
    # Convert correlation matrix to network edges
    for i, gene1 in enumerate(correlation_matrix.columns):
        for j, gene2 in enumerate(correlation_matrix.columns[i+1:], i+1):
            if abs(correlation_matrix.iloc[i, j]) > 0.8:  # Adjust threshold as needed
                coexp_network.add_edge(gene1, gene2, weight=correlation_matrix.iloc[i, j])

    # Step 5: Seek non-preserved modules in the control network
    non_preserved_modules = find_non_preserved_modules(coexp_network, ESA_G)

    # Step 6: Construct PPI network based on genes in the non-preserved modules
    ppi_network = construct_ppi_network(non_preserved_modules)

    # Step 7: Select top 20 genes with highest degree in PPI network
    top_genes = select_top_genes(ppi_network, 20)

    return top_genes

def find_non_preserved_modules(coexp_network, ESA_G):
    # Implement module preservation analysis using WGCNA or other methods
    # For simplicity, let's assume we're just identifying modules based on connected components
    modules = [list(component) for component in nx.connected_components(coexp_network)]
    # Calculate module Z statistics and identify non-preserved modules
    non_preserved_modules = []
    for module in modules:
        z_statistic = calculate_z_statistic(module, coexp_network, ESA_G)
        if z_statistic < 2:  # Adjust threshold as needed
            non_preserved_modules.append(module)
    return non_preserved_modules

def calculate_z_statistic(module, coexp_network, ESA_G):
    
    '''
    * Implement Z statistic calculation based on module density and connectivity.
    * For simplicity, we will calculate average correlation of genes in the module.
    '''
    
    correlations = []
    for i in range(len(module)):
        for j in range(i+1, len(module)):
            correlations.append(coexp_network[module[i]][module[j]]['weight'])
    avg_correlation = np.mean(correlations)
    
    # Calculate Z statistic
    z_statistic = (avg_correlation - np.mean(ESA_G.corr())) / np.std(ESA_G.corr())
    return z_statistic

def construct_ppi_network(non_preserved_modules):
    
    # We will construct a PPI network using some existing protein-protein interaction database
    ppi_network = nx.Graph()
    for module in non_preserved_modules:
        ppi_network.add_nodes_from(module)
    ppi_data = pd.read_csv("ppi_data.csv")
    for _, edge in ppi_data.iterrows():
        if edge['gene1'] in ppi_network.nodes and edge['gene2'] in ppi_network.nodes:
            ppi_network.add_edge(edge['gene1'], edge['gene2'], weight=edge['weight'])  # Adjust weight as needed
    return ppi_network

def select_top_genes(ppi_network, num_genes):
    
    # Get top genes based on degree centrality
    degree_centrality = nx.degree_centrality(ppi_network)
    sorted_genes = sorted(degree_centrality, key=degree_centrality.get, reverse=True)[:num_genes]
    return sorted_genes


In [115]:
def DMN_miRNA(GA, mRNA_miRNA_database):
    # Step 1: Extract miRNA set R from mRNA-miRNA databases
    R = extract_miRNA_set(mRNA_miRNA_database)
    
    # Step 2: Construct bipartite graph network G_miR
    G_miR = construct_bipartite_graph(R, GA)
    
    # Step 3: Construct R' using set cover algorithm
    R_prime = set_cover_algorithm(G_miR, GA)
    
    return R_prime

def extract_miRNA_set(mRNA_miRNA_database):
    # Implement extraction of miRNA set R from mRNA-miRNA databases
    # Placeholder: assuming mRNA_miRNA_database is a list of miRNAs
    return set(mRNA_miRNA_database)

def construct_bipartite_graph(R, GA):
    # Implement construction of bipartite graph network G_miR
    G_miR = nx.Graph()
    for miRNA in R:
        for gene in GA:
            # Check if miRNA targets gene (placeholder: randomly assign edges)
            if hash(miRNA) % 2 == hash(gene) % 2:
                G_miR.add_edge(miRNA, gene)
    return G_miR

def set_cover_algorithm(G_miR, GA):
    # Implement set cover algorithm to obtain minimum set R' of effective miRNAs
    # Placeholder: return all miRNAs with at least one gene target
    R_prime = set()
    for miRNA, gene in G_miR.edges:
        R_prime.add(miRNA)
    return R_prime



In [116]:
def check_autism(sample_data):

    critical_genes = FA_gene(sample_data, ESA, ESC)
    effective_miRNAs = DMN_miRNA(critical_genes, mRNA_miRNA_database)
    if len(effective_miRNAs) > 0:
        return "Autism"
    else:
        return "No Autism"